In [2]:
import pandas as pd

In [10]:
base_file ="AviationData/"
airline_losses = pd.read_csv(base_file + "flight_reroutes.csv",index_col=0)
display(airline_losses)

,airline,flight_number,origin,destination,original_route,new_route,original_distance_km,new_distance_km,additional_distance_km,extra_fuel_cost_usd,delay_hours
date,,,,,,,,,,,
2026-02-28,Iran Air,IR9334,Los Angeles,Doha,Via Iran/Iraq corridor,Via Turkey–Kazakhstan northern route,4239,5887,1648,6633.95,4.5
2026-02-28,Turkish Airlines,TU7911,Paris,Dubai,Direct Tehran overfly,Via Ashgabat–Almaty route,6888,8697,1809,9543.08,3.1
2026-02-28,Turkish Airlines,TU907,London,Doha,Direct Gulf route,Via Jordan–Egypt alternate,11403,12416,1013,4268.36,4.2
2026-02-28,Air France,AI2443,Tehran,Istanbul,Via Persian Gulf,Via Indian Ocean southern route,7180,9863,2683,12945.15,2.1
2026-02-28,KLM,KL8000,Lahore,Doha,Via Strait of Hormuz,Via Red Sea–Suez route,6222,7574,1352,9305.36,3.1
...,...,...,...,...,...,...,...,...,...,...,...
2026-03-16,Turkish Airlines,TU5008,Dubai,Mumbai,Via Iran/Iraq corridor,Via Egypt–East Africa southern route,4100,6649,2549,13216.68,4.8
2026-03-16,Thai Airways,TH9694,Singapore,Dubai,Via Baghdad FIR,Via Ankara–Baku corridor,3969,5185,1216,4860.83,0.9
2026-03-16,EgyptAir,EG8171,Kuwait City,Mumbai,Via Iran/Iraq corridor,Via Turkey–Kazakhstan northern route,6853,8212,1359,6382.05,3.2


In [ ]:
# Group conflict_events by date and severity
conflict_events_daily = conflict_events.groupby(['date', 'severity']).size().reset_index(name='conflict_count')

# Group cancellations by date (or by date and severity if possible)
flight_cancellation_daily = flight_cancellation.groupby('date').size().reset_index(name='cancellation_count')

# Merge on date and severity (if severity is available in cancellations, otherwise just on date)
merged_df = pd.merge(conflict_events_daily, flight_cancellation_daily, on='date', how='left')

# Now you can group by severity
avg_cancellations_by_severity = merged_df.groupby('severity')['cancellation_count'].mean().reset_index()
print(avg_cancellations_by_severity)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.barplot(x='severity', y='cancellation_count', data=avg_cancellations_by_severity, palette='viridis')
plt.title('Average Flight Cancellations by Conflict Severity')
plt.ylabel('Average Cancellations')
plt.xlabel('Conflict Severity')
plt.tight_layout()
plt.show()

In [ ]:
# Group conflict_events by date and severity (no airline)
conflict_events_daily = conflict_events.groupby(['date', 'severity']).size().reset_index(name='conflict_count')

# Group cancellations by date and airline
flight_cancellation_daily = flight_cancellation.groupby(['date', 'airline']).size().reset_index(name='cancellation_count')

# Merge conflict info onto cancellations by date
merged_df = pd.merge(flight_cancellation_daily, conflict_events_daily, on='date', how='left')

# Now you have airline, date, severity, conflict_count, cancellation_count
prediction_df = merged_df[['date', 'airline', 'severity', 'conflict_count', 'cancellation_count']].copy()
prediction_df = prediction_df.dropna(subset=['conflict_count', 'cancellation_count'])

display(prediction_df)

In [ ]:
# Select relevant columns for prediction
prediction_df = merged_df[['date', 'severity', 'conflict_count', 'cancellation_count']].copy()

# Drop rows with missing values if needed
prediction_df = prediction_df.dropna(subset=['conflict_count', 'cancellation_count'])

display(prediction_df)

In [ ]:
# Plotting severity zones by country using conflict_events dataset
import plotly.express as px

# Group by location and get the most frequent (mode) severity per location
country_severity = conflict_events.groupby('location')['severity'].agg(lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0]).reset_index()

# Custom color mapping for severity categories
color_discrete_map = {
    'Low': '#FFE5E5',        # very light red (almost pink)
    'Medium': '#FF9999',     # light red
    'High': '#FF4D4D',       # medium red
    'Very High': '#CC0000',  # strong red
    'Critical': '#660000'    # indigo (very dark)
}

# Ensure severity is string (if not already)
country_severity['severity'] = country_severity['severity'].astype(str)

fig = px.choropleth(
    country_severity,
    locations='location',
    locationmode='country names',
    color='severity',
    color_discrete_map=color_discrete_map,
    title='Severity Zones by Country',
    # You can adjust other options as needed
    hover_name='location',
    hover_data=['severity'],
    # scope='world',
    # projection='natural earth',
    # template='plotly_white',
    # etc.
    # See Plotly Express documentation for more options
)
fig.show()

In [ ]:
# Model 1: Predicting the Count of Flight Cancellations and Rerouted Flights

# 1. Merge and Prepare Data
# (Assumes conflict_events, flight_cancellation, and flight_reroutes are already loaded as DataFrames)

# Group conflict events by date and severity
conflict_count_daily = conflict_events.groupby(['date', 'severity']).size().reset_index(name='conflict_count')

# Group flight cancellations by date, airline, origin, destination
flight_cancellation_count = flight_cancellation.groupby(['date', 'airline', 'origin', 'destination']).size().reset_index(name='cancellation_count')

# Group rerouted flights by date, airline, origin, destination
flight_reroute_count = flight_reroutes.groupby(['date', 'airline', 'origin', 'destination']).size().reset_index(name='rerouted_count')

# Merge all together on date, airline, origin, destination
merged = pd.merge(flight_cancellation_count, conflict_count_daily, on='date', how='left')
merged = pd.merge(merged, flight_reroute_count, on=['date', 'airline', 'origin', 'destination'], how='left')

# Fill NaN rerouted_count with 0 (no reroute for that group)
merged['rerouted_count'] = merged['rerouted_count'].fillna(0)

# Drop rows with missing values in key columns
merged = merged.dropna(subset=['cancellation_count', 'conflict_count', 'severity'])

# 2. Encode Categorical Variables
categorical_cols = ['airline', 'origin', 'destination', 'severity']
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
], remainder='passthrough')

# 3. Feature Selection
X = merged[['airline', 'origin', 'destination', 'severity', 'conflict_count']]
y_cancel = merged['cancellation_count']
y_reroute = merged['rerouted_count']

# 4. Train-Test Split
X_train, X_test, y_cancel_train, y_cancel_test, y_reroute_train, y_reroute_test = train_test_split(
    X, y_cancel, y_reroute, test_size=0.2, random_state=42
)

# 5. Preprocess Features
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 6. Train Regression Models
cancel_model = RandomForestRegressor(n_estimators=100, random_state=42)
cancel_model.fit(X_train_processed, y_cancel_train)
reroute_model = RandomForestRegressor(n_estimators=100, random_state=42)
reroute_model.fit(X_train_processed, y_reroute_train)

# 7. Evaluate Models
from sklearn.metrics import mean_absolute_error, r2_score

y_cancel_pred = cancel_model.predict(X_test_processed)
y_reroute_pred = reroute_model.predict(X_test_processed)

print('Flight Cancellation Model:')
print('MAE:', mean_absolute_error(y_cancel_test, y_cancel_pred))
print('R2:', r2_score(y_cancel_test, y_cancel_pred))

print('\nRerouted Flights Model:')
print('MAE:', mean_absolute_error(y_reroute_test, y_reroute_pred))
print('R2:', r2_score(y_reroute_test, y_reroute_pred))


#### Model Improvement: Data Exploration, Feature Engineering, and Alternative Approaches

The initial model results indicate poor predictive power. Let's try the following steps to improve the model:

1. **Data Exploration:**
   - Visualize the distribution of target variables and features.
   - Check for outliers and class imbalance.
2. **Feature Engineering:**
   - Add new features (e.g., day of week, region, airline type).
   - Aggregate/group data differently if needed.
3. **Alternative Modeling:**
   - Try a classification approach (e.g., predict if cancellation > 0).
   - Use cross-validation for robust evaluation.

The next cells will walk through these steps.

In [ ]:
# 1. Data Exploration: Visualize Target Distributions and Check for Imbalance
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].hist(y_cancel, bins=20, color='skyblue', edgecolor='black')
axs[0].set_title('Flight Cancellation Count Distribution')
axs[0].set_xlabel('Cancellations')
axs[0].set_ylabel('Frequency')
axs[1].hist(y_reroute, bins=20, color='salmon', edgecolor='black')
axs[1].set_title('Rerouted Flights Count Distribution')
axs[1].set_xlabel('Rerouted Flights')
axs[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print('Unique cancellation counts:', y_cancel.value_counts().sort_index())
print('Unique rerouted counts:', y_reroute.value_counts().sort_index())

In [ ]:
# 2. Feature Engineering: Add Day of Week and Binary Target for Classification
merged['date'] = pd.to_datetime(merged['date'], errors='coerce')
merged['day_of_week'] = merged['date'].dt.day_name()
merged['is_cancelled'] = (merged['cancellation_count'] > 0).astype(int)

# Show new features
display(merged[['date', 'airline', 'origin', 'destination', 'severity', 'conflict_count', 'cancellation_count', 'day_of_week', 'is_cancelled']].head())

In [ ]:
# 3. Alternative Modeling: Classification (Predict if Any Cancellation)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Prepare features for classification
X_cls = merged[['airline', 'origin', 'destination', 'severity', 'conflict_count', 'day_of_week']]
y_cls = merged['is_cancelled']

# One-hot encode categorical features
preprocessor_cls = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['airline', 'origin', 'destination', 'severity', 'day_of_week'])
], remainder='passthrough')

X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)
X_cls_train_processed = preprocessor_cls.fit_transform(X_cls_train)
X_cls_test_processed = preprocessor_cls.transform(X_cls_test)

# Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_cls_train_processed, y_cls_train)
y_cls_pred = clf.predict(X_cls_test_processed)

print('Classification Accuracy:', accuracy_score(y_cls_test, y_cls_pred))
print('Confusion Matrix:\n', confusion_matrix(y_cls_test, y_cls_pred))
print('Classification Report:\n', classification_report(y_cls_test, y_cls_pred))

In [ ]:
# 8. What-if Scenario Example
what_if = pd.DataFrame([{
    'airline': 'Oman Air',
    'origin': 'Dubai',
    'destination': 'London',
    'severity': 'Critical',
    'conflict_count': 3
}])

# Preprocess the what-if scenario using the same pipeline as training
what_if_processed = preprocessor.transform(what_if)

# Predict cancellations and rerouted flights
pred_cancel = cancel_model.predict(what_if_processed)
pred_reroute = reroute_model.predict(what_if_processed)

print("\nWhat-if scenario prediction:")
print(f"Predicted cancellations: {pred_cancel[0]:.2f}")
print(f"Predicted rerouted flights: {pred_reroute[0]:.2f}")

In [ ]:
### Summary: Flight Cancellation Prediction Process

This section outlines the process used to predict flight cancellations using the merged dataset containing columns: `date`, `airline`, `cancellation_count`, `severity`, and `conflict_count`.

**1. Data Preparation:**
- Start with a DataFrame containing relevant features: date, airline, cancellation_count, severity, and conflict_count.
- Encode categorical variables (such as airline and severity) using one-hot encoding or label encoding to make them suitable for machine learning models.
- Select features (e.g., airline, severity, conflict_count) as predictors and set `cancellation_count` as the target variable.

**2. Feature Engineering:**
- Optionally, create new features such as extracting the day of the week from the date or grouping airlines by region to enhance model performance.

**3. Train-Test Split:**
- Split the data into training and testing sets (commonly 80% for training and 20% for testing) to evaluate model performance on unseen data.

**4. Model Selection:**
- Choose appropriate regression models (e.g., Linear Regression, Random Forest Regressor) to predict the number of flight cancellations.

**5. Model Training:**
- Train the selected model(s) using the training data.

**6. Prediction:**
- Use the trained model to predict `cancellation_count` on the test set.

**7. Evaluation:**
- Assess model performance using metrics such as Mean Absolute Error (MAE), Mean Squared Error (MSE), and R² score.

**8. Interpretation:**
- Analyze feature importance to understand which factors most influence flight cancellations.

This process enables data-driven prediction and analysis of flight cancellations, supporting better decision-making and operational planning in the aviation sector.

In [ ]:
# Predicting the actual number of cancelled and rerouted flights (not binary)
# Group by relevant columns to get counts per group

group_cols = ['date', 'airline', 'origin', 'destination', 'severity']

# Count cancellations per group
cancellation_counts = flight_cancellation.groupby(group_cols).size().reset_index(name='cancellation_count')
# Count reroutes per group
reroute_counts = flight_reroutes.groupby(group_cols).size().reset_index(name='reroute_count')

# Merge counts
merged_counts = pd.merge(cancellation_counts, reroute_counts, on=group_cols, how='outer')
merged_counts['cancellation_count'] = merged_counts['cancellation_count'].fillna(0)
merged_counts['reroute_count'] = merged_counts['reroute_count'].fillna(0)

# Print columns to debug passengers column name
print('flight_reroutes columns:', list(flight_reroutes.columns))

def get_passenger_col(df):
    for col in df.columns:
        if 'passenger' in col.lower():
            return col
    return None

route_cols = ['date', 'airline', 'origin', 'destination']

# Distance features always come from reroutes
dist_features = flight_reroutes.groupby(route_cols).agg({
    'original_distance_km': 'mean',
    'new_distance_km': 'mean'
}).reset_index()

# Passenger feature: try reroutes first, then fallback to cancel_agg, then flight_cancellation
passenger_col = get_passenger_col(flight_reroutes)

if passenger_col is not None:
    print('Using passengers column from flight_reroutes:', passenger_col)
    pass_features = (
        flight_reroutes.groupby(route_cols)[passenger_col]
        .mean()
        .reset_index()
        .rename(columns={passenger_col: 'passengers_affected'})
    )
elif 'cancel_agg' in globals() and set(route_cols + ['passengers_affected']).issubset(cancel_agg.columns):
    print("Using passengers_affected from existing 'cancel_agg'")
    pass_features = cancel_agg[route_cols + ['passengers_affected']].copy()
else:
    fallback_col = get_passenger_col(flight_cancellation)
    if fallback_col is None:
        raise KeyError(
            "No passengers column found in flight_reroutes or flight_cancellation."
        )
    print('Using passengers column from flight_cancellation:', fallback_col)
    pass_features = (
        flight_cancellation.groupby(route_cols)[fallback_col]
        .mean()
        .reset_index()
        .rename(columns={fallback_col: 'passengers_affected'})
    )

dist_pass_features = pd.merge(dist_features, pass_features, on=route_cols, how='left')

merged_counts = pd.merge(merged_counts, dist_pass_features, on=route_cols, how='left')

# Features and targets
features = ['severity', 'original_distance_km', 'new_distance_km', 'passengers_affected']
target_cancel = 'cancellation_count'
target_reroute = 'reroute_count'

# Drop rows with missing values in features
merged_counts = merged_counts.dropna(subset=features)

X = merged_counts[features]
y_cancel = merged_counts[target_cancel]
y_reroute = merged_counts[target_reroute]

In [ ]:
# Analysis the conflict_event and flight_cancellation dataset 
display(conflict_events.head())
display(flight_cancellation.head())

In [ ]:
# Group conflict events by date and severity (no airlines)
conflict_count_daily = conflict_events.groupby(['date','severity']).size().reset_index(name="conflict_count")

# Group flight cancellation by date, airlines and passengers_affected
flight_cancellation_count = flight_cancellation.groupby(['date','airline']).size().reset_index(name="cancellation_count")

# merging the cancellation and conflict even count dataframes
conflict_cancellation = pd.merge(flight_cancellation_count,conflict_count_daily,on='date',how='left')

display(conflict_cancellation)

In [ ]:
new_conflict_cancellation = conflict_cancellation[['date','airline','cancellation_count','severity','conflict_count']].copy()
new_conflict_cancellation = new_conflict_cancellation.dropna(subset=['cancellation_count','conflict_count']).reset_index(drop=True)
display(new_conflict_cancellation)

In [ ]:
#visualizing the dataset

plt.figure(figsize=(8,6))
sns.barplot(new_conflict_cancellation,x="severity",y=new_conflict_cancellation['cancellation_count'].index,palette="viridis", errorbar= None)
plt.xlabel("Severity")
plt.ylabel("No.of.flight cancelled")
plt.title("Flight cancellation by Conflict severity")
plt.tight_layout()
plt.show()

In [ ]:
# Group by location and get the most frequent (mode) severity per location
country_severity = conflict_events.groupby('location')['severity'].agg(lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0]).reset_index()

# Custom color mapping for severity categories
color_discrete_map = {
    'Low': '#FFE5E5',        # very light red (almost pink)
    'Medium': '#FF9999',     # light red
    'High': '#FF4D4D',       # medium red
    'Very High': '#CC0000',  # strong red
    'Critical': '#660000'    # indigo (very dark)
}

# Ensure severity is string (if not already)
country_severity['severity'] = country_severity['severity'].astype(str)

fig = px.choropleth(
    country_severity,
    locations='location',
    locationmode='country names',
    color='severity',
    color_discrete_map=color_discrete_map,
    title='Severity Zones by Country',
    # You can adjust other options as needed
    hover_name='location',
    hover_data=['severity'],
    # scope='world',
    # projection='natural earth',
    # template='plotly_white',
    # etc.
    # See Plotly Express documentation for more options
)
fig.show()

In [ ]:
# Improved geographic map of most affected routes with legend
# This example assumes you have a DataFrame 'airport_coords' with columns: 'airport', 'lat', 'lon'
# If you don't have this, you can use a public airport coordinates dataset and merge by IATA code

# Example: create airport_coords DataFrame (replace with your real data or merge from a CSV)
# airport_coords = pd.read_csv('airport_coordinates.csv')
# airport_coords = airport_coords[['IATA', 'Latitude', 'Longitude']].rename(columns={'IATA': 'airport', 'Latitude': 'lat', 'Longitude': 'lon'})

# For demonstration, we'll create dummy coordinates (replace with real data for accuracy)
# Merge coordinates for origin and destination
route_counts = (
    flight_cancellation.groupby(['origin', 'destination'])['flight_number']
    .count()
    .reset_index(name='cancellation_count')
    .sort_values(by='cancellation_count', ascending=False)
)
top_routes = route_counts.head(20)

# Dummy coordinates for demonstration (replace with real airport coordinates)
import numpy as np
import plotly.graph_objects as go
import plotly.colors as pc
np.random.seed(0)
unique_airports = pd.unique(top_routes[['origin', 'destination']].values.ravel())
airport_coords = pd.DataFrame({
    'airport': unique_airports,
    'lat': np.random.uniform(10, 60, size=len(unique_airports)),
    'lon': np.random.uniform(-130, 60, size=len(unique_airports))
})

# Merge coordinates for plotting
routes_plot = top_routes.merge(airport_coords, left_on='origin', right_on='airport')
routes_plot = routes_plot.merge(airport_coords, left_on='destination', right_on='airport', suffixes=('_origin', '_dest'))

# Assign a unique color to each route
route_colors = pc.qualitative.Plotly * ((len(routes_plot) // len(pc.qualitative.Plotly)) + 1)

fig = go.Figure()

# Add lines for each route, each with a different color and legend
for idx, row in routes_plot.iterrows():
    fig.add_trace(go.Scattergeo(
        lon=[row['lon_origin'], row['lon_dest']],
        lat=[row['lat_origin'], row['lat_dest']],
        mode='lines',
        line=dict(width=2 + row['cancellation_count'] / routes_plot['cancellation_count'].max() * 6, color=route_colors[idx]),
        opacity=0.7,
        name=f"{row['origin']} → {row['destination']}",
        showlegend=True if idx < 20 else False,  # Only show legend for top 20
        hoverinfo='text',
        text=f"{row['origin']} → {row['destination']}<br>Cancellations: {row['cancellation_count']}"
    ))

# Add airport points
fig.add_trace(go.Scattergeo(
    lon=airport_coords['lon'],
    lat=airport_coords['lat'],
    mode='markers',
    marker=dict(size=6, color='blue'),
    text=airport_coords['airport'],
    name='Airports',
    showlegend=True
))

fig.update_layout(
    title_text='Top 20 Most Affected Flight Routes (Cancellations)',
    showlegend=True,
    legend=dict(title='Routes', font=dict(size=10), orientation='v', x=1.05, y=1),
    geo=dict(
        projection_type='natural earth',
        showland=True,
        landcolor='rgb(243, 243, 243)',
        countrycolor='rgb(204, 204, 204)',
    ),
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

# Note: For real analysis, replace the dummy airport_coords with actual airport latitude/longitude data.

In [ ]:
print("="*200)
print("Heatmap: Airline Cancellations vs. Reason")
print("="*200)
# Create a heatmap for airline cancellations vs reason
if 'reason' in flight_cancellation.columns:
    pivot_table = pd.pivot_table(
        flight_cancellation,
        values='flight_number',
        index='airline',
        columns='reason',
        aggfunc='count',
        fill_value=0
    )
    plt.figure(figsize=(14,10))
    # Revert to the previous color map: 'YlGnBu'
    sns.heatmap(pivot_table, annot=True, fmt='d', cmap='YlGnBu', linewidths=0.5, cbar_kws={'label': 'Number of Cancellations'})
    plt.title('Heatmap: Airline Cancellations vs. Reason')
    plt.xlabel('Reason for Cancellation')
    plt.ylabel('Airline')
    plt.tight_layout()
    plt.show()
else:
    print("Column 'reason' not found in flight_cancellation.")


print("="*200)
print("Flight cancellation on the war dates")
print("="*200)
# No of cancelled flight on war dates
cancelled_flight = flight_cancellation.groupby('date')['flight_number'].count()

# line graph represent the flight cancellation
plt.figure(figsize=(8,6))
sns.lineplot(x=cancelled_flight.index, y = cancelled_flight.values, color="red")
sns.scatterplot(x=cancelled_flight.index, y = cancelled_flight.values, color ="blue")
plt.fill_between(cancelled_flight.index,cancelled_flight.values, alpha=0.4)
plt.grid(True, alpha=0.3)
plt.xticks(rotation = 90)
plt.xlabel("dates")
plt.ylabel("no.of.flight cancelled")
plt.title("flight cancellation on the war dates")
plt.tight_layout()
plt.show()

print("="*200)
print("KPI dashboard for flight cancellation")
print("="*200)

# --- KPI Dashboard: Total Cancellations and Passengers Affected ---
total_cancellations = flight_cancellation['flight_number'].nunique()

#Total passengers affected

if 'passengers_affected' in flight_cancellation.columns:
    total_passengers_affected = flight_cancellation["passengers_affected"].sum()
else:
    total_passengers_affected = None

# Display as KPI cards
kpi_md = f"""
<div style='display:flex; gap:40px;'>
  <div style='background:#e3f2fd; padding:20px; border-radius:10px; min-width:220px; text-align:center;'>
    <h2 style='color:#1976d2;'>✈️ Total Cancellations</h2>
    <p style='font-size:2em; color:#0d47a1; margin:0;'>{total_cancellations:,}</p>
  </div>
  <div style='background:#fce4ec; padding:20px; border-radius:10px; min-width:220px; text-align:center;'>
    <h2 style='color:#c2185b;'>👥 Passengers Affected</h2>
    <p style='font-size:2em; color:#880e4f; margin:0;'>{total_passengers_affected:,}</p>
  </div>
</div>
"""
display(Markdown(kpi_md))

print("="*200)
print("Flights Cancelled and Passengers Affected per Airline")
print("="*200)

#cancelled flights vs passenger affected for each airlines

#calculating flight cancelled per airlines
cancelled_flight = flight_cancellation.groupby('airline')['flight_number'].count().reset_index(name='cancelled_flights')

#calculating passenger affected per airlines
passenger_affect = flight_cancellation.groupby('airline')['passengers_affected'].sum().reset_index(name='passengers_affected')

#airline_summary

airline_summary = pd.merge(cancelled_flight, passenger_affect, on='airline')

fig, ax1 = plt.subplots(figsize=(15,8))

# Bar plot for flights cancelled
ax1.bar(airline_summary['airline'], airline_summary['cancelled_flights'], color='skyblue', label='Flights Cancelled')
ax1.set_xlabel('Airline')
ax1.set_ylabel('Flights Cancelled', color='skyblue')
ax1.tick_params(axis='y', labelcolor='skyblue')
plt.xticks(rotation=90)

# Second y-axis for passengers affected
ax2 = ax1.twinx()
ax2.plot(airline_summary['airline'], airline_summary['passengers_affected'], color='crimson', marker='o', label='Passengers Affected')
ax2.set_ylabel('Passengers Affected', color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')

plt.title('Flights Cancelled and Passengers Affected per Airline')
ax1.grid(True,axis='both',alpha=0.3)
ax2.grid(True,axis='both',alpha=0.3)
plt.tight_layout()
plt.show()

print("="*200)
print("Flight Cancellations by Country (Origin vs Destination)")
print("="*200)


# Side-by-side bar chart for origin and destination country cancellations
import numpy as np

# Recompute country-level counts from flight_cancellation to avoid column-name mismatches
origin_country_counts = (
    flight_cancellation.groupby('origin_country')['flight_number']
    .count()
    .reset_index(name='origin_count')
)

destination_country_counts = (
    flight_cancellation.groupby('destination_country')['flight_number']
    .count()
    .reset_index(name='destination_count')
)

countries = list(
    set(origin_country_counts['origin_country']).union(
        set(destination_country_counts['destination_country'])
    )
)
countries.sort()

origin_counts = (
    origin_country_counts.set_index('origin_country')
    .reindex(countries, fill_value=0)['origin_count']
)
destination_counts = (
    destination_country_counts.set_index('destination_country')
    .reindex(countries, fill_value=0)['destination_count']
)

x = np.arange(len(countries))  # label locations
width = 0.35  # width of the bars
plt.figure(figsize=(12,7))
plt.bar(x - width/2, origin_counts, width, label='Origin Country')
plt.bar(x + width/2, destination_counts, width, label='Destination Country')
plt.xticks(x, countries, rotation=90)
plt.xlabel('Country')
plt.ylabel('Number of Cancellations')
plt.title('Flight Cancellations by Country (Origin vs Destination)')
plt.legend()
plt.tight_layout()
plt.show()

print("="*200)
print("Flight Cancellations top 10 routes")
print("="*200)


# flight cancellation for each origin
origin_count = flight_cancellation.groupby('origin')['flight_number'].count().reset_index(name='origin_count')
# flight cancellation for each destination
destination_count = flight_cancellation.groupby('destination')['flight_number'].count().reset_index(name='destination_count')

# calculating flight cancellation for origin and destination
origin_destination = pd.merge(origin_count, destination_count, left_on='origin', right_on='destination', how='outer')

# Map Flow (Sankey) for Top 15 Routes by Cancellation Count with Different Colors
import plotly.graph_objects as go
import plotly.colors as pc

# Prepare data: top 15 most affected routes
route_counts = (
    flight_cancellation.groupby(['origin', 'destination'])['flight_number']
    .count()
    .reset_index(name='cancellation_count')
    .sort_values(by='cancellation_count', ascending=False)
)
top_routes = route_counts.head(15)

# Create unique list of airports for node labels
unique_airports = list(set(top_routes['origin']).union(set(top_routes['destination'])))
label_index = {airport: idx for idx, airport in enumerate(unique_airports)}

# Sankey diagram source, target, value
sources = top_routes['origin'].map(label_index)
targets = top_routes['destination'].map(label_index)
values = top_routes['cancellation_count']

# Assign a different color for each route
route_colors = pc.qualitative.Plotly * ((len(top_routes) // len(pc.qualitative.Plotly)) + 1)
link_colors = route_colors[:len(top_routes)]

fig = go.Figure(data=[
    go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=unique_airports,
            color="blue"
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors
        )
    )
])

fig.update_layout(title_text="Top 15 Most Affected Flight Routes (Cancellations) - Flow Map (Colored Routes)", font_size=12)
fig.show()

# Calculate average passengers affected per cancellation (overall)
if 'passengers_affected' in flight_cancellation.columns:
    avg_passengers_per_cancel = flight_cancellation['passengers_affected'].sum() / flight_cancellation['flight_number'].nunique()
    print(f"Average passengers affected per cancelled flight (overall): {avg_passengers_per_cancel:.2f}")
else:
    print("Column 'passengers_affected' not found in flight_cancellation.")

# Average passengers affected per cancellation by airline
if 'passengers_affected' in flight_cancellation.columns:
    airline_avg = flight_cancellation.groupby('airline').apply(lambda df: df['passengers_affected'].sum() / df['flight_number'].nunique()).reset_index(name='avg_passengers_per_cancel')
    #display(airline_avg.sort_values('avg_passengers_per_cancel', ascending=False),airline_avg.head(10))
    
    # Visualization
    plt.figure(figsize=(12,6))
    sns.lineplot(x='airline', y='avg_passengers_per_cancel', data=airline_avg.sort_values('avg_passengers_per_cancel', ascending=False))
    sns.scatterplot(
        x='airline',
        y='avg_passengers_per_cancel',
        data=airline_avg.sort_values('avg_passengers_per_cancel', ascending=False),
        color='orange'
    )
    plt.xticks(rotation=90)
    plt.ylabel('Avg Passengers per Cancellation')
    plt.xlabel('Airline')
    plt.title('Top 10 Airlines: Avg Passengers Affected per Cancelled Flight')
    plt.tight_layout()
    plt.show()
else:
    print("Column 'passengers_affected' not found in flight_cancellation.")